In [1]:
# use ck_env 

In [1]:
import climkern as ck # https://github.com/tyfolino/climkern
import xarray as xr
from utils.config import model

In [2]:
renaming_dict = {'tas':'TS', 
                 'ta':'T',
                 'hus':'Q',
                 'ps':'PS',
                 'rsus':'FSUS',
                 'rsds':'FSDS',
                 #'ptp':'TROP_P'
                }

In [3]:
def get_all_feedbacks_and_save(ctrl, pert, labctrl, labpert):
    LR, Planck = ck.calc_T_feedbacks(
                ctrl.T.assign_attrs({'units':'K'}),
                ctrl.TS.assign_attrs({'units':'K'}), 
                ctrl.PS.assign_attrs({'units':'Pa'}), 
                pert.T.assign_attrs({'units':'K'}), 
                pert.TS.assign_attrs({'units':'K'}), 
                pert.PS.assign_attrs({'units':'Pa'}), 
                pert_trop=None, 
                fixRH=False, 
                kern="HadGEM3-GA7.1"
                        )
    q_lw,q_sw = ck.calc_q_feedbacks(ctrl.Q.assign_attrs({'units':'1'}),
                                ctrl.T.assign_attrs({'units':'K'}),
                                ctrl.PS.assign_attrs({'units':'Pa'}),
                                pert.Q.assign_attrs({'units':'1'}),
                                pert.PS.assign_attrs({'units':'Pa'}),
                                pert_trop=None,
                                kern="HadGEM3-GA7.1",
                                method=1)
    
    alb = ck.calc_alb_feedback(ctrl.FSUS,
                           ctrl.FSDS,
                           pert.FSUS,
                           pert.FSDS,
                           kern="HadGEM3-GA7.1")
    ## also get all the clear-sky versions, for the cloud masking term
    LR_cs,Planck_cs = ck.calc_T_feedbacks(ctrl.T.assign_attrs({'units':'K'}),
                                        ctrl.TS.assign_attrs({'units':'K'}), 
                                        ctrl.PS.assign_attrs({'units':'Pa'}), 
                                        pert.T.assign_attrs({'units':'K'}), 
                                        pert.TS.assign_attrs({'units':'K'}), 
                                        pert.PS.assign_attrs({'units':'Pa'}), 
                                        pert_trop=None, 
                                        fixRH=False, 
                                        kern="HadGEM3-GA7.1",sky="clear-sky")
    
    q_lw_cs,q_sw_cs = ck.calc_q_feedbacks(ctrl.Q.assign_attrs({'units':'1'}),
                                        ctrl.T.assign_attrs({'units':'K'}),
                                        ctrl.PS.assign_attrs({'units':'Pa'}),
                                        pert.Q.assign_attrs({'units':'1'}),
                                        pert.PS.assign_attrs({'units':'Pa'}),
                                        pert_trop=None,
                                        kern="HadGEM3-GA7.1",method=1,sky="clear-sky")
    
    alb_cs = ck.calc_alb_feedback(ctrl.FSUS,ctrl.FSDS,
                               pert.FSUS,pert.FSDS,
                               kern="HadGEM3-GA7.1",sky="clear-sky")

    ## finally, add strat T and strat q, since i am using IRF not F:
    T_strat =  ck.calc_strato_T(ctrl.T.assign_attrs({'units':'K'}),
                                pert.T.assign_attrs({'units':'K'}),
                                pert.PS.assign_attrs({'units':'Pa'}),
                                pert_trop=None, kern="HadGEM3-GA7.1", sky='all-sky')
    
    q_strat = ck.calc_strato_q(ctrl.Q.assign_attrs({'units':'1'}),
                               ctrl.T.assign_attrs({'units':'K'}),
                               pert.Q.assign_attrs({'units':'1'}),
                               pert.PS.assign_attrs({'units':'Pa'}),
                               pert_trop=None, kern="HadGEM3-GA7.1", 
                               sky='all-sky', method=1)
    
    T_strat_cs =  ck.calc_strato_T(ctrl.T.assign_attrs({'units':'K'}),
                                pert.T.assign_attrs({'units':'K'}),
                                pert.PS.assign_attrs({'units':'Pa'}),
                                pert_trop=None, kern="HadGEM3-GA7.1",
                                sky='clear-sky')
    
    q_strat_cs = ck.calc_strato_q(ctrl.Q.assign_attrs({'units':'1'}),
                               ctrl.T.assign_attrs({'units':'K'}),
                               pert.Q.assign_attrs({'units':'1'}),
                               pert.PS.assign_attrs({'units':'Pa'}),
                               pert_trop=None, kern="HadGEM3-GA7.1", 
                               sky='clear-sky', method=1)

    ## save to a dataset:
    ds = xr.merge([LR.to_dataset(name='LR'),
                   Planck.to_dataset(name='Planck'),
                   (q_sw+q_lw).to_dataset(name='WV'),
                   alb.to_dataset(name='Alb'),
                   LR_cs.to_dataset(name='LR_cs'),
                   Planck_cs.to_dataset(name='Planck_cs'),
                   (q_sw_cs+q_lw_cs).to_dataset(name='WV_cs'),
                   alb_cs.to_dataset(name='Alb_cs'),
                   T_strat.to_dataset(name='T_strat'),
                   T_strat_cs.to_dataset(name='T_strat_cs'),
                   (q_strat[0]+q_strat[1]).to_dataset(name='q_strat'), # this is sum of LW and SW
                   (q_strat_cs[0]+q_strat_cs[1]).to_dataset(name='q_strat_cs'), # this is sum of LW and SW
                   (pert.TS-ctrl.TS).to_dataset(name='delta_TS')])
    
    ds.to_netcdf('intermediate_outputs/feedbacks/{a}_{b}_feedbacks.nc'.format(a=labctrl, b=labpert))

In [4]:
#ctrl_ref, pert_ref = ck.tutorial_data("ctrl"), ck.tutorial_data("pert")

ds_ssp245 = xr.open_dataset('intermediate_outputs/for_kernel_decomp/{s}_{m}.nc'.format(
                        s='ssp245', m=model)).rename({'month':'time'}).rename(renaming_dict)
ds_ssp245_baseline = xr.open_dataset('intermediate_outputs/for_kernel_decomp/{s}_{m}.nc'.format(
                        s='ssp245_baseline', m=model)).rename({'month':'time'}).rename(renaming_dict)
ds_arise = xr.open_dataset('intermediate_outputs/for_kernel_decomp/{s}_{m}.nc'.format(
                        s='ARISE', m=model)).rename({'month':'time'}).rename(renaming_dict)

In [9]:
# run for warming (future ssp245 - baseline ssp245)
get_all_feedbacks_and_save(ctrl=ds_ssp245_baseline, pert=ds_ssp245, labctrl='baseline', labpert='ssp245')

In [10]:
# run for SAI (ARISE - SSP245)
get_all_feedbacks_and_save(ctrl=ds_ssp245, pert=ds_arise, labctrl='ssp245', labpert='ARISE')

In [5]:
# run for SAI (ARISE - SSP245)
get_all_feedbacks_and_save(ctrl=ds_ssp245_baseline, pert=ds_arise, labctrl='baseline', labpert='ARISE')

In [ ]:
### old from below

In [4]:
LR, Planck = ck.calc_T_feedbacks(
                ctrl.T.assign_attrs({'units':'K'}),
                ctrl.TS.assign_attrs({'units':'K'}), 
                ctrl.PS.assign_attrs({'units':'Pa'}), 
                pert.T.assign_attrs({'units':'K'}), 
                pert.TS.assign_attrs({'units':'K'}), 
                pert.PS.assign_attrs({'units':'Pa'}), 
                pert_trop=None, 
                fixRH=True, 
                kern="HadGEM3-GA7.1"
                        )

In [5]:
q_lw,q_sw = ck.calc_q_feedbacks(ctrl.Q.assign_attrs({'units':'1'}),
                                ctrl.T.assign_attrs({'units':'K'}),
                                ctrl.PS.assign_attrs({'units':'Pa'}),
                                pert.Q.assign_attrs({'units':'1'}),
                                pert.PS.assign_attrs({'units':'Pa'}),
                                pert_trop=None,
                                kern="HadGEM3-GA7.1",
                                method=1)

In [6]:
alb = ck.calc_alb_feedback(ctrl.FSUS,
                           ctrl.FSDS,
                           pert.FSUS,
                           pert.FSDS,
                           kern="HadGEM3-GA7.1")


In [7]:
## also get all the clear-sky versions, for the cloud masking term
LR_cs,Planck_cs = ck.calc_T_feedbacks(ctrl.T.assign_attrs({'units':'K'}),
                                    ctrl.TS.assign_attrs({'units':'K'}), 
                                    ctrl.PS.assign_attrs({'units':'Pa'}), 
                                    pert.T.assign_attrs({'units':'K'}), 
                                    pert.TS.assign_attrs({'units':'K'}), 
                                    pert.PS.assign_attrs({'units':'Pa'}), 
                                    pert_trop=None, 
                                    fixRH=True, 
                                    kern="HadGEM3-GA7.1",sky="clear-sky")

q_lw_cs,q_sw_cs = ck.calc_q_feedbacks(ctrl.Q.assign_attrs({'units':'1'}),
                                    ctrl.T.assign_attrs({'units':'K'}),
                                    ctrl.PS.assign_attrs({'units':'Pa'}),
                                    pert.Q.assign_attrs({'units':'1'}),
                                    pert.PS.assign_attrs({'units':'Pa'}),
                                    pert_trop=None,
                                    kern="HadGEM3-GA7.1",method=1,sky="clear-sky")

alb_cs = ck.calc_alb_feedback(ctrl.FSUS,ctrl.FSDS,
                           pert.FSUS,pert.FSDS,
                           kern="HadGEM3-GA7.1",sky="clear-sky")

In [15]:
## finally, add strat T and strat q, since i am using IRF not F:
T_strat =  ck.calc_strato_T(ctrl.T.assign_attrs({'units':'K'}),
                            pert.T.assign_attrs({'units':'K'}),
                            pert.PS.assign_attrs({'units':'Pa'}),
                            pert_trop=None, kern="HadGEM3-GA7.1", sky='all-sky')

q_strat = ck.calc_strato_q(ctrl.Q.assign_attrs({'units':'1'}),
                           ctrl.T.assign_attrs({'units':'K'}),
                           pert.Q.assign_attrs({'units':'1'}),
                           pert.PS.assign_attrs({'units':'Pa'}),
                           pert_trop=None, kern="HadGEM3-GA7.1", 
                           sky='all-sky', method=1)

T_strat_cs =  ck.calc_strato_T(ctrl.T.assign_attrs({'units':'K'}),
                            pert.T.assign_attrs({'units':'K'}),
                            pert.PS.assign_attrs({'units':'Pa'}),
                            pert_trop=None, kern="HadGEM3-GA7.1",
                            sky='clear-sky')

q_strat_cs = ck.calc_strato_q(ctrl.Q.assign_attrs({'units':'1'}),
                           ctrl.T.assign_attrs({'units':'K'}),
                           pert.Q.assign_attrs({'units':'1'}),
                           pert.PS.assign_attrs({'units':'Pa'}),
                           pert_trop=None, kern="HadGEM3-GA7.1", 
                           sky='clear-sky', method=1)

In [17]:
## save to a dataset:
ds = xr.merge([LR.to_dataset(name='LR'),
               Planck.to_dataset(name='Planck'),
               (q_sw+q_lw).to_dataset(name='WV'),
               alb.to_dataset(name='Alb'),
               LR_cs.to_dataset(name='LR_cs'),
               Planck_cs.to_dataset(name='Planck_cs'),
               (q_sw_cs+q_lw_cs).to_dataset(name='WV_cs'),
               alb_cs.to_dataset(name='Alb_cs'),
               T_strat.to_dataset(name='T_strat'),
               T_strat_cs.to_dataset(name='T_strat_cs'),
               (q_strat[0]+q_strat[1]).to_dataset(name='q_strat'), # this is sum of LW and SW
               (q_strat_cs[0]+q_strat_cs[1]).to_dataset(name='q_strat_cs'), # this is sum of LW and SW
               (pert.TS-ctrl.TS).to_dataset(name='delta_TS')])

ds.to_netcdf('intermediate_outputs/feedbacks.nc')